In [ ]:
!pip install line_profiler
%load_ext line_profiler

In [47]:
from collections import defaultdict
from tqdm import tqdm
from cs336_basics.pretokenize import pre_tokenize_parallel


def train_bpe(input_path, vocab_size, special_tokens):
    merges = []
    vocab = [bytes([i]) for i in range(256)]
    vocab += [s.encode("utf-8") for s in special_tokens]
    init_vocab_len = len(vocab)

    print("Pretokening .......")
    token_counter = pre_tokenize_parallel(input_path, special_tokens)

    print("Counting pairs .......")
    pair_counter = defaultdict(int)
    for token, count in token_counter.items():
        for i in range(len(token) - 1):
            pair = token[i : i + 2]
            pair_counter[pair] += count

    print("Training BPE .......")
    for epoch in tqdm(range(init_vocab_len, vocab_size)):
        max_pair = max(pair_counter.keys(), key=lambda pair: (pair_counter[pair], pair))
        merged_pair = max_pair[0] + max_pair[1]
        merges.append(max_pair)
        new_counter = {}
        for token, count in token_counter.items():
            if max_pair[0] not in token or max_pair[1] not in token:
                new_counter[token] = count
                continue
            new_token = []
            i = 0
            while i < len(token):
                pair = token[i : i + 2]
                if pair == max_pair:
                    # Update the pair_counts
                    pair_counter[pair] -= count
                    if i > 0:
                        left_pair = (new_token[-1], pair[0])
                        pair_counter[left_pair] -= count
                        pair_counter[(left_pair[0], merged_pair)] += count
                    if i < len(token) - 2:
                        right_pair = token[i + 1 : i + 3]
                        pair_counter[right_pair] -= count
                        pair_counter[(merged_pair, right_pair[1])] += count
                    new_token.append(merged_pair)
                    i += 2
                else:
                    new_token.append(token[i])
                    i += 1
            new_counter[tuple(new_token)] = count
        token_counter = new_counter
        vocab.append(merged_pair)

    vocab = {i: x for i, x in enumerate(vocab)}
    return vocab, merges

In [ ]:
special_tokens = ["<|endoftext|>"]
input_path = "../data/TinyStoriesV2-GPT4-valid.txt"
vocab_size = 1000
vocab, merges = train_bpe(input_path, vocab_size, special_tokens)

Pretokening .......


100%|██████████| 8/8 [00:01<00:00,  4.00it/s]


Counting pairs .......
Training BPE .......


 74%|███████▍  | 742/1000 [00:08<00:02, 89.53it/s] 


In [33]:
%lprun -f train_bpe train_bpe(input_path, vocab_size, special_tokens)

Pretokening .......


100%|██████████| 1/1 [00:00<00:00, 11.13it/s]


Initializing pairs .......
Training BPE .......


 74%|███████▍  | 742/1000 [00:21<00:07, 34.41it/s]


Timer unit: 1e-09 s

Total time: 18.7026 s
File: /var/folders/gq/bts52kyn0v72cpz5c5489984006lvv/T/ipykernel_26407/908302985.py
Function: train_bpe at line 6

Line #      Hits         Time  Per Hit   % Time  Line Contents
     6                                           def train_bpe(input_path, vocab_size, special_tokens):
     7         1       1000.0   1000.0      0.0      merges = []
     8         1      39000.0  39000.0      0.0      vocab = [bytes([i]) for i in range(256)]
     9         1       4000.0   4000.0      0.0      vocab += [s.encode("utf-8") for s in special_tokens]
    10                                           
    11         1     507000.0 507000.0      0.0      print("Pretokening .......")
    12         1  229274000.0 2.29e+08      1.2      token_counter = pre_tokenize_parallel(input_path, special_tokens)
    13                                           
    14         1     148000.0 148000.0      0.0      print("Initializing pairs .......")
    15         1    

In [50]:
def new_train_bpe(input_path, vocab_size, special_tokens):
    merges = []
    vocab = [bytes([i]) for i in range(256)]
    vocab += [s.encode("utf-8") for s in special_tokens]

    print("Pretokening .......")
    token_counter = pre_tokenize_parallel(input_path, special_tokens)

    print("Initializing pairs .......")
    merges = []
    vocab = [bytes([i]) for i in range(256)]
    vocab += [s.encode("utf-8") for s in special_tokens]
    init_vocab_len = len(vocab)

    token_pair_set = dict()
    pair_counter = defaultdict(int)
    for token, count in token_counter.items():
        s = set()
        for i in range(len(token) - 1):
            pair = token[i : i + 2]
            pair_counter[pair] += count
            s.add(pair)
        token_pair_set[token] = s

    print("Training BPE .......")
    for epoch in tqdm(range(init_vocab_len, vocab_size)):
        max_pair = max(pair_counter, key=lambda p: (pair_counter[p], p))
        mp0, mp1 = max_pair
        merged_pair = mp0 + mp1
        merges.append(max_pair)

        for token, count in list(token_counter.items()):
            if mp0 not in token or mp1 not in token:
                continue  # no occurrence — token stays unchanged in dict
            modify = False
            new_token = []
            i = 0
            n = len(token)
            while i < n:
                if i + 1 < n and token[i] == mp0 and token[i + 1] == mp1:
                    modify = True
                    if new_token:  # left-neighbor update
                        left = new_token[-1]
                        lp = (left, mp0)
                        c = pair_counter[lp] - count
                        if c:
                            pair_counter[lp] = c
                        else:
                            del pair_counter[lp]
                        pair_counter[(left, merged_pair)] += count
                    if i + 2 < n:  # right-neighbor update
                        right = token[i + 2]
                        rp = (mp1, right)
                        c = pair_counter[rp] - count
                        if c:
                            pair_counter[rp] = c
                        else:
                            del pair_counter[rp]
                        pair_counter[(merged_pair, right)] += count
                    new_token.append(merged_pair)
                    i += 2
                else:
                    new_token.append(token[i])
                    i += 1

            # ── in-place update (no new_counter) ──────────────────────────
            if modify:
                new_key = tuple(new_token)
                del token_counter[token]
                del token_pair_set[token]
                token_counter[new_key] = token_counter.get(new_key, 0) + count
                token_pair_set[new_key] = {
                    (new_token[j], new_token[j + 1]) for j in range(len(new_token) - 1)
                }



        del pair_counter[max_pair]  # max_pair gone from all tokens; remove stale count
        vocab.append(merged_pair)

    vocab = {i: x for i, x in enumerate(vocab)}
    return vocab, merges

In [48]:
special_tokens = ["<|endoftext|>"]
input_path = "../data/TinyStoriesV2-GPT4-valid.txt"
vocab_size = 10000
new_vocab, new_merges = new_train_bpe(input_path, vocab_size, special_tokens)

Pretokening .......


100%|██████████| 8/8 [00:01<00:00,  4.54it/s]


Initializing pairs .......
Training BPE .......


  9%|▊         | 848/9742 [00:02<00:25, 345.00it/s]


KeyboardInterrupt: 

In [25]:
for m, n in zip(merges, new_merges):
    if m != n:
        print("not the same")
        print(m, n)
        break

not the same
(b' ', b'a') (b'h', b'e')


In [49]:
from tests.common import gpt2_bytes_to_unicode

input_path = "../tests/fixtures/corpus.en"
vocab, merges = train_bpe(
    input_path=input_path,
    vocab_size=500,
    special_tokens=["<|endoftext|>"],
)

# Path to the reference tokenizer vocab and merges
reference_vocab_path = "../tests/fixtures/train-bpe-reference-vocab.json"
reference_merges_path = "../tests/fixtures/train-bpe-reference-merges.txt"

# Compare the learned merges to the expected output merges
gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
with open(reference_merges_path, encoding="utf-8") as f:
    gpt2_reference_merges = [tuple(line.rstrip().split(" ")) for line in f]
    reference_merges = [
        (
            bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
            bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
        )
        for merge_token_1, merge_token_2 in gpt2_reference_merges
    ]
assert merges == reference_merges

Pretokening .......


100%|██████████| 1/1 [00:00<00:00, 12.36it/s]


Counting pairs .......
Training BPE .......


100%|██████████| 243/243 [00:00<00:00, 810.91it/s]


In [46]:
len(vocab)

499

In [51]:
from tests.common import gpt2_bytes_to_unicode

input_path = "../tests/fixtures/corpus.en"
vocab_new, merges_new = new_train_bpe(
    input_path=input_path,
    vocab_size=500,
    special_tokens=["<|endoftext|>"],
)

# Path to the reference tokenizer vocab and merges
reference_vocab_path = "../tests/fixtures/train-bpe-reference-vocab.json"
reference_merges_path = "../tests/fixtures/train-bpe-reference-merges.txt"

# Compare the learned merges to the expected output merges
gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
with open(reference_merges_path, encoding="utf-8") as f:
    gpt2_reference_merges = [tuple(line.rstrip().split(" ")) for line in f]
    reference_merges = [
        (
            bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
            bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
        )
        for merge_token_1, merge_token_2 in gpt2_reference_merges
    ]
assert merges == reference_merges

Pretokening .......


100%|██████████| 1/1 [00:00<00:00, 14.43it/s]


Initializing pairs .......
Training BPE .......


100%|██████████| 243/243 [00:00<00:00, 701.87it/s]
